# Phase 4 — Load Cleaned Data into PostgreSQL

This notebook loads the cleaned Parquet file into a PostgreSQL database
called `db_delays`. From this point on, all analytical queries run
against the database using SQL.

**Input:** `data/cleaned/db_clean.parquet`
**Output:** Table `train_delays` in the `db_delays` PostgreSQL database

In [8]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine.url import URL
from dotenv import load_dotenv
import os

# Load secrets from .env into environment variables
load_dotenv("../.env")

# Read each variable
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Build the connection URL safely (handles special characters in password)
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name,
)

# Create the engine
engine = create_engine(connection_url)

# Test the connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("✅ Connected to PostgreSQL successfully.")
    print("Server version:", result.scalar())

✅ Connected to PostgreSQL successfully.
Server version: PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit


In [9]:
# Load the cleaned data
df_clean = pd.read_parquet("../data/cleaned/db_clean.parquet")
print(f"Loaded {len(df_clean):,} rows from parquet.")

# Write to PostgreSQL
table_name = "train_delays"
df_clean.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
    method="multi"
)
print(f"✅ Loaded into PostgreSQL table: {table_name}")

Loaded 1,850,002 rows from parquet.


KeyboardInterrupt: 

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
 result = conn.execute(text("SELECT COUNT(*) FROM train_delays;"))
 row_count = result.scalar()
 print(f"Rows in train_delays: {row_count:,}")

 result = conn.execute(text("SELECT * FROM train_delays LIMIT 3;"))
 for row in result:
  print(row)

Rows in train_delays: 1,850,002
('1573967790757085557-2407072312-14', '20', 'Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|Alsdorf Poststraße|Alsdorf-Mariadorf|Alsdorf-Kellersberg|Alsdorf-Annapark|Alsdorf-Busch|Herzogenrath-August-Schmidt-Platz|Herzogenrath-Alt-Merkstein|Herzogenrath|Kohlscheid|Aachen West|Aachen Schanz', 8000001, 2, 'Aachen Hbf', 'Nordrhein-Westfalen', 'Aachen', 52064, 6.091499, 50.7678, datetime.datetime(2024, 7, 8, 0, 0), datetime.datetime(2024, 7, 8, 0, 1), datetime.datetime(2024, 7, 8, 0, 3), datetime.datetime(2024, 7, 8, 0, 4), 3, 3, None, 'on_time', 'on_time', 0, 0, 'Monday', 7, 'July', 2024, 'Summer', 'On time (≤5 min)')
('7157250219775883918-2407072120-25', '1', 'Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Kurl|Dortmund-Scharnhorst|Dortmund Hbf|Bochum Hbf|Wattenscheid|Essen Hbf|Mülheim(Ruhr)Hbf|Duisburg Hbf|Dü ... (33 characters truncated) ... |Düsseldorf-Benrath|Leverkusen Mitte|Köln-Mülheim|Köln Messe/Deutz|Köln Hbf|Köln-Ehrenfeld|Horrem|Düren|Langerwehe|Esc

In [ ]:
import pandas as pd

with open("../sql/01_overall_punctuality.sql", "r") as f:
 query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,total_arrivals,on_time_count,delayed_count,on_time_percentage,avg_delay_minutes,median_delay_minutes,max_delay_minutes
0,1850002,1739049,110953,94.0,1.31,0.0,159


## Q1 Finding (preliminary)

- 94.0% of train stops in this dataset arrive within DB's official 6-minute on-time window.
- My prediction was 70% — I was off by 24 percentage points in the pessimistic direction.
- Median delay is 0 minutes; mean is 1.31 minutes (heavily right-skewed).
- Max observed delay: 159 minutes (~2.6 hours).

**Important caveats:**
- Sample period is one week (July 7–14, 2024) — not representative of full year.
- Dataset includes regional and S-Bahn trains, which are more punctual than long-distance ICEs.
- DB's published ~80% punctuality figure is for long-distance trains only — not directly comparable.

In [10]:
with open("../sql/02_delays_by_category.sql", "r") as f:
 query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,category,total_arrivals,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,1,20079,2.30,1.0,88.70,2.43
1,2,38456,2.25,0.0,88.66,2.90
2,3,173374,1.70,0.0,92.46,1.48
3,4,354913,1.13,0.0,94.78,1.00
4,5,293178,1.40,0.0,93.13,1.38


## Q2 Finding

- Major hubs (category 1-2) are 5-6 percentage points less punctual
than small stations (category 4): ~89% vs ~95% on-time.
- Severely delayed trips (>15 min) are roughly 2-3x more common at
major hubs than at smaller stations.
- The pattern is monotonic from category 1 → 4, then slightly worse at
category 5 — possibly because category 5 stations include route
termini where delays often originate.
- My prediction was correct: bottlenecks accumulate delay.

In [11]:
with open("../sql/03_delays_by_hour.sql", "r") as f:
    query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,hour,total_arrivals,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,0,25630,1.40,0.0,93.73,1.62
1,1,12108,1.30,0.0,94.47,1.70
2,2,2980,1.36,0.0,95.03,2.08
3,3,2271,0.52,0.0,98.19,0.22
4,4,18643,0.32,0.0,99.20,0.23
5,5,46781,0.64,0.0,98.10,0.36
6,6,60426,0.84,0.0,97.06,0.58
7,7,52657,0.96,0.0,96.55,0.67
8,8,45875,1.19,0.0,94.84,0.65
9,9,42699,1.30,0.0,94.29,0.86


## Q3 Finding

- **The day gets progressively worse, not peaks-and-valleys.** On-time
 rate slides from 99% at 4 AM to 89% at 6 PM, then recovers overnight.
- **Best hour: 4 AM** (99.20% on-time, 0.32 min average delay).
- **Worst hour: 6 PM** (89.27% on-time, 2.26% severely delayed).
- **Evening rush (17-19) is significantly worse than morning rush (7-9).**
 Morning trains are still close to early-day cleanliness; evening
 trains carry the day's accumulated delays.
- My prediction (twin peaks at morning and evening rush) was only half
 right: I correctly identified evening rush as bad, but morning rush
 is actually relatively punctual.
- Practical takeaway for travelers: book early-morning trains when
 possible. Avoid arrivals in the 17:00–19:00 window if your schedule
 has no buffer.

**Cross-link to Q2:** The same phenomenon (delay accumulation) explains
both the major-hub effect and the evening-degradation effect.

**Caveat for Q3:** Hours 0-7 in the dataset include Monday, Tuesday,
Wednesday, AND a partial Thursday (until 8 AM Thursday). Hours 8-23
include only Mon-Wed. This minor imbalance doesn't change the overall
shape of the day pattern but is worth noting.

In [12]:
with open("../sql/04_delays_by_weekday.sql", "r") as f:
    query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,weekday_name,total_arrivals,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,Monday,275949,1.31,0.0,93.85,1.20
1,Tuesday,277737,1.52,0.0,92.62,1.54
2,Wednesday,279183,1.47,0.0,93.09,1.34
3,Thursday,47031,0.91,0.0,96.51,0.92
4,Sunday,100,2.22,1.0,91.00,3.00


In [13]:
verification_query = """
SELECT
    DATE(arrival_plan) AS date,
    weekday_name,
    COUNT(*) AS row_count,
    MIN(arrival_plan) AS earliest,
    MAX(arrival_plan) AS latest
FROM train_delays
GROUP BY DATE(arrival_plan), weekday_name
ORDER BY date;
"""

date_check = pd.read_sql(verification_query, engine)
date_check

,date,weekday_name,row_count,earliest,latest
0,2024-07-07,Sunday,100,2024-07-07 23:37:00,2024-07-07 23:59:00
1,2024-07-08,Monday,275949,2024-07-08 00:00:00,2024-07-08 23:59:00
2,2024-07-09,Tuesday,277737,2024-07-09 00:00:00,2024-07-09 23:59:00
3,2024-07-10,Wednesday,279183,2024-07-10 00:00:00,2024-07-10 23:59:00
4,2024-07-11,Thursday,47031,2024-07-11 00:00:00,2024-07-11 07:59:00


In [14]:
with open("../sql/04_delays_by_weekday.sql", "r") as f:
    query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,date,weekday_name,total_arrivals,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,2024-07-08,Monday,275949,1.31,0.0,93.85,1.20
1,2024-07-09,Tuesday,277737,1.52,0.0,92.62,1.54
2,2024-07-10,Wednesday,279183,1.47,0.0,93.09,1.34


## Q4 Finding (with strong caveats)

- Across the 3 full days available (Mon Jul 8, Tue Jul 9, Wed Jul 10):
 - Monday: 93.85% on-time
 - Tuesday: 92.62% on-time (worst)
 - Wednesday: 93.09% on-time
- The differences are small (~1 percentage point) and consistent across
 metrics (lower on-time %, higher avg delay, more severe delays on Tue).

**Important limitation:** With only one of each weekday in the dataset,
this is a single-day comparison — not a true weekly pattern. The 1pp
gap could reflect real weekly variation OR a one-day disruption
(weather, maintenance, signal issue) on that specific Tuesday.

To make a credible weekday-pattern claim, the analysis would need
multiple weeks of data. As-is, the finding is best stated as:
"On these 3 specific days, Tuesday had slightly more delays than
Monday or Wednesday."

In [15]:
with open("../sql/05_worst_stations.sql", "r") as f:
    query = f.read()

result_df = pd.read_sql(query, engine)
result_df

,station,state,category,total_arrivals,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,Würzburg Hbf,Bayern,2,145,7.46,3.0,66.90,16.55
1,Sangerhausen,Sachsen-Anhalt,4,163,7.14,2.0,61.96,14.72
2,Steinau (Straße),Hessen,5,129,6.74,1.0,66.67,17.05
3,Wächtersbach,Hessen,4,155,6.54,1.0,66.45,16.77
4,Schlüchtern,Hessen,4,144,6.42,1.0,66.67,17.36
5,Varel (Oldb),Niedersachsen,5,120,6.10,0.0,58.33,13.33
6,Regensburg Hbf,Bayern,2,208,5.98,3.5,60.10,9.13
7,Bad Soden-Salmünster,Hessen,5,140,5.86,1.0,69.29,16.43
8,Sigmaringen,Baden-Württemberg,5,102,5.83,1.0,75.49,14.71
9,Menden (Rheinl),Nordrhein-Westfalen,5,129,5.79,3.0,58.91,9.30


## Q6 Finding (and a surprise)

- **My prediction was wrong.** I expected major hubs (Frankfurt Hbf,
 München Hbf, Köln Hbf) to dominate the worst-station list. They don't
 appear at all. The list is mostly small to medium stations.

- **The real pattern is geographic, not size-based.** 7 of the top 20
 worst stations are in Hessen, mostly along the Frankfurt-Fulda
 corridor (Steinau, Wächtersbach, Schlüchtern, Bad Soden-Salmünster,
 Flieden, Neuhof, Langenselbold). This suggests a corridor-level
 disruption during the sample week, not a structural "small-station"
 problem.

- **Worst single station:** Würzburg Hbf (Bayern) — only 66.90% on-time
 vs the 94% network average — and it's a major hub, but a regional
 outlier rather than reflecting the whole "category 2" pattern.

- **Reconciliation with Q2:** Q2 said major hubs are *on average*
 worse than smaller stations. Q6 shows the *worst individual outliers*
 are mostly smaller stations. Both are true: hubs have consistently
 mediocre performance; small stations are mostly fine but a few are
 catastrophically bad on specific corridors.

**Caveat:** With only 100-230 arrivals per top-20 station over one week,
these rankings could shift substantially with more data. The
finding "Hessen has a problem corridor this week" is robust;
"Steinau is the 3rd worst station in Germany" is not.

In [18]:
from sqlalchemy import text

with open("../sql/06_route_length.sql", "r") as f:
    query = f.read()

with engine.connect() as conn:
    result_df = pd.read_sql(text(query), conn)

result_df

,route_length_bucket,total_arrivals,avg_stops_in_bucket,avg_delay_min,median_delay_min,on_time_pct,severely_delayed_pct
0,1. Short (1-3 stops),173079,2.0,0.72,0.0,96.68,0.71
1,2. Medium (4-8 stops),253870,5.9,1.19,0.0,94.54,1.19
2,3. Long (9-15 stops),236314,11.7,1.60,0.0,92.70,1.51
3,4. Very long (16+ stops),216737,21.6,2.00,0.0,90.07,1.81


## Q5 Finding

- **Strong, clean relationship between route length and delay.**
- Short routes (1-3 stops): 96.68% on-time, avg 0.72 min delay
- Very long routes (16+ stops): 90.07% on-time, avg 2.00 min delay
- That's a 6.6 percentage point drop in on-time rate as routes grow.
- Severe delays (>15 min) are 2.5x more common on very long routes.
- My prediction was correct in direction (longer = more delay) but
 I overestimated the magnitude.

**Connecting insight:** This is the same delay-accumulation phenomenon
seen in Q2 (delays compound at major hubs) and Q3 (delays compound
through the day). All three findings point to one underlying story:
**the network has limited slack to absorb disruption, so delays
propagate across time, distance, and network position.**